In [ ]:
import os
import sys
import xarray as xr
import netCDF4 as nc
import pandas
import numpy as np
import glob
import pandas as pd
import scipy.io as sio

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
from matplotlib.gridspec import GridSpec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo
import seaborn as sns

# settings
%config InlineBackend.figure_format = 'retina'

# Proxy data. Override with PROXY_DATA_DIR; see config/paths.env.example.
# The default is the repo-relative tree, so this notebook runs with no setup provided it is launched from the repo root.
dpath0 = os.environ.get('PROXY_DATA_DIR', 'data/raw')
dpath1 = f'{dpath0}/DSDP-480-479/age_model'
# Repo-tracked external data (LR04 benthic stack).
extern = 'data/external'
# save figs here
opath = os.environ.get('FIG_OUTPUT_DIR', 'outputs')
os.makedirs(opath, exist_ok=True)

**Bacon age models**

In [ ]:
# DSDP-480 depths
dsdp480_depths=pd.read_excel(f'{dpath1}/sample_depths_480.xlsx').depth.values # cm

# DSDP-480 bacon inputs
dsdp480_input=pd.read_csv(f'{dpath0}/Bacon_runs/DSDP480/DSDP480.csv')
xerr480 = dsdp480_input.error.values  # x-axis error

# DSDP-480 bacon ensemble.
# CANONICAL age model for DSDP-480 is Bacon_runs/DSDP480 — see DATA_MANIFEST.md section 2.
# This ensemble is the one behind DSDP480_165_ages.txt, which the MATLAB pipeline
# (dDwax_data_processing_d480_d479.m) uses to assign sample ages: its column medians match
# that file's `median` to 0.5 yr, and its 2.5/97.5 column percentiles match `min`/`max` to 0.5 yr.
dsdp480_mcmc=pd.read_csv(f'{dpath0}/Bacon_runs/DSDP480/DSDP480_mcmc_new.csv', header=None)
ens_num=np.linspace(1,len(dsdp480_mcmc),len(dsdp480_mcmc))
dims = ['ensemble_number','depth']
coords = {'ensemble_number': ens_num,
          'depth': dsdp480_depths}
dsdp480_agedepth=xr.DataArray(dsdp480_mcmc.values, dims=dims, coords=coords) # each row is an age model
median480=dsdp480_agedepth.median(axis=0)

# Confidence bands: percentiles of the ensemble at each depth, ordered [2.5, 16, 84, 97.5].
# Percentiles rather than indices into a sorted array so this stays correct at any ensemble width
p480 = np.nanpercentile(dsdp480_mcmc.values, [2.5, 16, 84, 97.5], axis=0)


# ---------------------------------------------------------------------------------------------
# PUT THE TIE POINTS ON THE SAME TIMESCALE AS THE AGE MODEL BEFORE PLOTTING THEM.
#
# DSDP480.csv stores each date in whatever units Bacon expects as INPUT, and those units are not
# the same for every row. The `cc` column selects the calibration curve:
#
#   cc = 0  ->  `age` is ALREADY a calendar age. Bacon takes it as given. (rows 0-5, 12-16:
#               the Murray/Barron magnetic-susceptibility ties, the SH82 and d479 tie points)
#   cc = 2  ->  `age` is an UNCALIBRATED radiocarbon age. Bacon calibrates it internally against
#               the marine curve, shifted by the local reservoir offset `dR` (300 +/- 20 yr).
#               (rows 6-11: the six Keigwin & Jones planktic dates)
#
# The age-depth model Bacon returns is in CALENDAR years BP. So plotting a cc=2 row's raw `age`
# against that model puts a radiocarbon year on a calendar-year axis. 

# The cc=2 markers plotting locations are MODEL-DEPENDENT. They sit on the median line by construction
# and are NOT an independent check of the fit; they only show which depths carry radiocarbon control.
# The cc=0 markers are still drawn at their own input ages and remain genuine independent checks of the model.
d480_udepth, _first = np.unique(dsdp480_depths, return_index=True)  # 2390 cm is duplicated

def _on_agemodel(per_depth_values, depths=None):
    """Interpolate a per-depth age-model quantity onto depths (cm).

    Defaults to the Bacon tie-point depths; pass `depths` for anything else.
    """
    if depths is None:
        depths = dsdp480_input.depth.values
    return np.interp(depths, d480_udepth, per_depth_values[_first])

is_c14 = (dsdp480_input.cc.values == 2)          # the rows Bacon calibrated internally
# Calendar-age position of every tie point:
# Bacon's posterior median for cc=2, the input age for cc=0 (which is already calendar)
d480_tie_age = np.where(is_c14, _on_agemodel(median480.values), dsdp480_input.age.values)
# Error bars: Bacon's 95% interval for cc=2, the reported +/-2 sigma for cc=0.
d480_tie_lo = np.where(is_c14, _on_agemodel(p480[0]), d480_tie_age - xerr480 * 2)
d480_tie_hi = np.where(is_c14, _on_agemodel(p480[3]), d480_tie_age + xerr480 * 2)
# matplotlib wants asymmetric xerr as [[distance below], [distance above]]
d480_tie_xerr = np.vstack([d480_tie_age - d480_tie_lo, d480_tie_hi - d480_tie_age])

In [ ]:
# DSDP-480 has two samples at 2390 cm, so `depth` is not a unique index and .interp() fails
# on median480 directly. Drop the duplicate to get an interpolable series — the two ensemble
# columns at 2390 cm carry identical medians (39244.2789 yr), so which one is dropped is moot.
# This is why the round trip through pandas exists; do not "simplify" it away.
new_median = median480.to_series().reset_index().drop_duplicates(subset='depth').set_index('depth').to_xarray()

# DSDP-480/479 splice tie points, interpolated off the canonical age model. age (yr), depth (cm).
d480_479_tiepoint = [[float(new_median.interp(depth=4307).to_array()), 4307], # pollen tie point
                     [float(new_median.interp(depth=4596).to_array()), 4596]] # dDwax tie point
#[110134.71, 4351],

In [ ]:
# DSDP-479 depths
dsdp479_depths=pd.read_excel(f'{dpath1}/sample_depths_479.xlsx').depth.values # cm

# DSDP-479 bacon inputs
dsdp479_input=pd.read_csv(f'{dpath0}/Bacon_runs/DSDP479/DSDP479.csv')
xerr479 = dsdp479_input.error.values  # x-axis error

# DSDP-479 bacon ensemble. its column medians match DSDP479_113_ages.txt to 0.5 yr.
dsdp479_mcmc=pd.read_csv(f'{dpath0}/Bacon_runs/DSDP479/DSDP479_mcmc.csv', header=None)
ens_num=np.linspace(1,len(dsdp479_mcmc),len(dsdp479_mcmc))
dims = ['ensemble_number','depth']
coords = {'ensemble_number': ens_num,
          'depth': dsdp479_depths}
dsdp479_agedepth=xr.DataArray(dsdp479_mcmc.values, dims=dims, coords=coords) # each row is an age model
median479=dsdp479_agedepth.median(axis=0)

# Confidence bands, ordered [2.5, 16, 84, 97.5] — same convention as p480 above.
p479 = np.nanpercentile(dsdp479_mcmc.values, [2.5, 16, 84, 97.5], axis=0)

In [ ]:
# DSDP-479/480 splice tie points, interpolated off the DSDP-479 age model. age (yr), depth (cm).
# 479 depths are unique, so median479 is directly interpolable — no drop_duplicates needed here.
d479_480_tiepoint = [[float(median479.interp(depth=3476)), 3476], # pollen tie point
                     [float(median479.interp(depth=4351)), 4351]] # dDwax tie point

**Resolved (2026-08-07): the ¹⁴C points not lining up was a plotting bug, not an age-model problem.**

`DSDP480.csv` mixes two timescales. Rows with `cc=0` carry calendar ages; rows with `cc=2` — the
six Keigwin & Jones planktic dates — carry *uncalibrated radiocarbon* ages that Bacon converts
internally using the marine curve and the reservoir offset `dR = 300`. The age-depth model comes
back in calendar years, so drawing the raw `cc=2` ages against it put radiocarbon years on a
calendar-year axis.

The tell was that the offset grew monotonically with depth (+790 yr at 1051 cm → +3008 yr at
1806 cm) while the `cc=0` ties over the same interval, which need no conversion, sat within ~85 yr
of the model. A genuinely wrong age model would have missed both kinds; only the ones needing a
unit conversion were off.

The ¹⁴C markers are now drawn at Bacon's calibrated calendar age for their depth. **Caveat:** that
makes them model-dependent — they lie on the median line by construction and no longer serve as an
independent check. They mark which depths carry radiocarbon control. The `cc=0` markers are still
plotted at their own input ages and remain independent checks. See the age-model cell for the full
reasoning and for what an independent check would require.

In [ ]:
line_kw={'ls':'-', 'lw':2}
err_line_kw={'ls':':', 'lw':0.75, 'color':'k'}
scat_kw = {'s': 40, 'edgecolors':'k', 'alpha':1, 'clip_on': False, 'zorder':100}
# Filled markers carry a black edge. A marker drawn at the age the MODEL returned (rather than
# the age the model was given) is unfilled: a thin x / + in panels A and B, an open circle in
# panel C. For thin x/+ the stroke colour comes from `color`, so xmark_kw sets no edgecolors.
xmark_kw = {'s': 55, 'linewidths':1.8, 'alpha':1, 'clip_on': False, 'zorder':101}
xtie_kw  = {**scat_kw, 's': 75}   # filled X / P tie-points read too small at s=40
pmark_kw = {**xmark_kw, 's': 70}  # a thin '+' reads smaller than a thin 'x' at equal s
tri_kw   = {**scat_kw, 's': 62}   # ditto the 14C triangles
open_kw  = {'s': 46, 'facecolors':'none', 'edgecolors':'red', 'linewidths':1.6, 'alpha':1, 'clip_on': False, 'zorder':101}
tkw = {'axis':'both', 'direction':'in', 'labelsize': 10}
title_text_kw={'size':14, 'weight':'bold', 'color':'k', 'va':'center'}
label_text_kw={'size':11, 'weight':'bold', 'color':'firebrick', 'ha':'center', 'va':'bottom'} #'backgroundcolor':'white', 
axis_text_kw={'weight':'normal', 'size':11, 'color':'k'}
legend_kw = {'loc':3, 'fontsize':8, 'labelcolor':'k', 'frameon':False}

fig = plt.figure(figsize=(9,6))

# DSDP-480
ax = plt.subplot(121)
# 95% confidence interval
ax.plot(p480[0], dsdp480_agedepth.depth, **err_line_kw, label='_Hidden')
ax.plot(p480[3], dsdp480_agedepth.depth, **err_line_kw, label='_Hidden')
# 2-sigma shading
ax.fill_betweenx(dsdp480_agedepth.depth,
                 p480[0],
                 p480[3],
                 color='k', edgecolor='none', alpha=0.1, label='2$\sigma$ confidence')
# 1-sigma shading
ax.fill_betweenx(dsdp480_agedepth.depth,
                 p480[1],
                 p480[2],
                 color='k', edgecolor='none', alpha=0.2, label='1$\sigma$ confidence')               
# median line
ax.plot(median480, dsdp480_agedepth.depth, '-', c='k', lw=1, label='median')
# ms tie-points
plt.errorbar(d480_tie_age[1:6], dsdp480_input.iloc[1:6].depth.values, 
             xerr=d480_tie_xerr[:, 1:6], yerr=0, fmt='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(d480_tie_age[1:6], dsdp480_input.iloc[1:6].depth.values,
            marker='s', color='lightblue', **scat_kw, label='M.S. tie-point')
# planktic 14C, drawn at Bacon's calibrated (calendar) age -- NOT the raw 14C age in the csv.
# See the age-model cell for why, and for what this does and does not demonstrate.
plt.errorbar(d480_tie_age[6:12], dsdp480_input.iloc[6:12].depth.values, 
             xerr=d480_tie_xerr[:, 6:12], yerr=0, fmt='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(d480_tie_age[6:12], dsdp480_input.iloc[6:12].depth.values,
            marker='^', color='white', **tri_kw, label='$^{14}$C$_{planktic}$, calibrated')
# benthic d18O ties. Rows 12-13 are SH-1/SH-2 (correlated to Shackleton & Hall 1982); row 16 is
# this study's own sample at 4839 cm, which had no legend entry at all before. Same evidence type
# and same role in the model, so one symbol and one legend entry. Row 16 is the same sample that
# appears in panel C.
_d18o_ties = [12, 13, 16]
plt.errorbar(d480_tie_age[_d18o_ties], dsdp480_input.iloc[_d18o_ties].depth.values, 
             xerr=d480_tie_xerr[:, _d18o_ties], yerr=0, fmt='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(d480_tie_age[_d18o_ties], dsdp480_input.iloc[_d18o_ties].depth.values,
            marker='o', color='lightblue', **scat_kw, label='$\delta^{18}$O$_{benthic}$ tie-point')
# Cross-core ties at the age they were GIVEN to Bacon. Row 14 (4307 cm) is the pollen
# correlation and row 15 (4596 cm) the dDwax one -- previously both were drawn as one group
# labelled 'dD_C30 tie', which hid the fact that 4307 is a pollen tie.
plt.errorbar(d480_tie_age[14:16], dsdp480_input.iloc[14:16].depth.values, 
             xerr=d480_tie_xerr[:, 14:16], yerr=0, fmt='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(d480_tie_age[15], dsdp480_input.iloc[15].depth,
            marker='X', color='lightblue', **xtie_kw, label='$\delta$D$_{C30}$ tie-point')
plt.scatter(d480_tie_age[14], dsdp480_input.iloc[14].depth,
            marker='P', color='lightblue', **xtie_kw, label='pollen tie-point')
# The same two ties at their MODEL age
plt.scatter(d480_479_tiepoint[1][0], d480_479_tiepoint[1][1],
            marker='x', color='r', **xmark_kw, label='$\delta$D$_{C30}$ correlation 480↔479')
plt.scatter(d480_479_tiepoint[0][0], d480_479_tiepoint[0][1],
            marker='+', color='coral', **pmark_kw, label='pollen correlation 480↔479')
# Flip y-axis if needed (e.g., for depth increasing downward)
ax.invert_yaxis()
# Labels
ax.set(xlim=[1,137000], ylim=[5000,0])
ax.set_xlabel("Age (ka)", **axis_text_kw)
ax.set_ylabel("Core Depth (cm)")
ax.set_xticks([25000,50000,75000,100000,125000])
ax.set_xticklabels([25,50,75,100,125])
ax.xaxis.set_ticks_position('top')    # show ticks on top
ax.xaxis.set_label_position('top')    # show label on top
# Hide bottom ticks and label
ax.tick_params(bottom=False, labelbottom=False, **tkw)
ax.legend(**legend_kw)


# DSDP-479
ax = plt.subplot(122)
# 2-sigma bounds and shading
ax.plot(p479[0], dsdp479_agedepth.depth, **err_line_kw, label='_Hidden')
ax.plot(p479[3], dsdp479_agedepth.depth, **err_line_kw, label='_Hidden')
ax.fill_betweenx(dsdp479_agedepth.depth,
                 p479[0],
                 p479[3],
                 color='k', edgecolor='none', alpha=0.1, label='2$\sigma$ confidence interval')
# 1-sigma shading
ax.fill_betweenx(dsdp479_agedepth.depth,
                 p479[1],
                 p479[2],
                 color='k', edgecolor='none', alpha=0.2, label='1$\sigma$ confidence interval')
# median line
ax.plot(median479, dsdp479_agedepth.depth, '-', c='k', lw=1, label='median')
# d479-LR04 tie points
# DSDP-479's own tie points, at the ages given to Bacon. (dR is 0 for every row here, so the
# old `age - dR` was a no-op.)
plt.errorbar(dsdp479_input.age.values, dsdp479_input.depth.values, 
             xerr=xerr479*2, yerr=0, fmt='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(dsdp479_input.age.values, dsdp479_input.depth.values,
            marker='X', color='lightblue', **xtie_kw, label='$\delta$D$_{C30}$ tie-point')
# The DSDP-480 correlations at their MODEL age
plt.scatter(d479_480_tiepoint[1][0], d479_480_tiepoint[1][1],
            marker='x', color='r', **xmark_kw, label='$\delta$D$_{C30}$ correlation 480↔479')
plt.scatter(d479_480_tiepoint[0][0], d479_480_tiepoint[0][1],
            marker='+', color='coral', **pmark_kw, label='pollen correlation 480↔479')
# Flip y-axis if needed (e.g., for depth increasing downward)
ax.invert_yaxis()
# Labels
ax.set(xlim=[100000,160000], ylim=[5750,3450])
ax.set_xlabel("Age (ka)", **axis_text_kw)
ax.yaxis.set_label_position('right')
ax.set_ylabel("Core Depth (cm)", rotation=270, labelpad=20)
ax.set_xticks([100000,120000,140000,160000])
ax.set_xticklabels([100,120,140,160])
ax.xaxis.set_ticks_position('top')    # show ticks on top
ax.xaxis.set_label_position('top')    # show label on top
# Hide bottom ticks and label
ax.tick_params(bottom=False, labelbottom=False,
               left=False, labelleft=False,
               right=True, labelright=True, **tkw)

#plt.savefig(f'{opath}/dsdp_480-479.agemodel.pdf', bbox_inches='tight')

**Age model relevant proxy data on ages**

---
Shackleton & Hall, 1982 $\delta^{18}O$<br>
OXYGEN ISOTOPE STUDY OF CONTINUOUS SCRAPE SAMPLES FROM SITE 480<br>
DOI: 10.2973/dsdp.proc.64.165.1982

Keigwin & Jones, 1990 $\delta^{18}O$<br>
DOI:

Bryne et al., 1990 pollen<br>
DOI:

Lisiecki & Raymo, 2004 $\delta^{18}O$<br>
DOI:

In [ ]:
# dpath1 is defined in the setup cell at the top of the notebook.

# Shackleton & Hall d18O
filen=f'{dpath1}/ShackletonHall82_d18O.xlsx'
sh_d18o={}
depth=pd.read_excel(filen, sheet_name='Sheet1').depth.values
dat=pd.read_excel(filen, sheet_name='Sheet1').d18o.values
sh_d18o['dat'] = xr.DataArray(data=dat,
                              coords={'depth': depth},
                              dims='depth',
                              name='d18o')
sh_d18o['dat'].attrs['source'] = 'Shackleton_Hall_1982'

# Keigwin & Jones d18O
filen=f'{dpath1}/KeigwinJones90_d18O.xlsx'
kj_d18o={}
depth=pd.read_excel(filen, sheet_name='Sheet1').depth.values
dat=pd.read_excel(filen, sheet_name='Sheet1').d18o.values
kj_d18o['dat']=xr.DataArray(data=dat,
                            coords={'depth': depth},
                            dims='depth',
                            name='d18o')
kj_d18o['dat'].attrs['source'] = 'Keigwin_Jones_1990'

# Byrne et al. pollen
filen=f'{dpath1}/Byrne90_pollen.xlsx'
by_aj={}
depth=pd.read_excel(filen, sheet_name='Sheet1').depth.values
dat=pd.read_excel(filen, sheet_name='Sheet1').ArtemisiaJuniper.values
by_aj['dat']= xr.DataArray(data=dat,
                           coords={'depth': depth},
                           dims='depth',
                           name='ArtemisiaJuniper')
by_aj['dat'].attrs['source'] = 'Byrne_1990'

# LR04 benthic stack, read from data/external/lr04.mat
# `delob` columns are [age (ka), d18O (per mil), error].
delob = sio.loadmat(f'{extern}/lr04.mat')['delob']
lr04={}
lr04['age']=delob[:, 0]
lr04['dat']=delob[:, 1]

hol_d18o_max = 2.1
hol_d18o_min = 2.5

In [ ]:
tie_points = {}

# Benthic d18O from THIS study at 48.39 m = 4839 cm. The two values are REPLICATE analyses of
# one sample, so they share a single depth and a single age.
#
# WHICH AGE: the model's, not the tie's. 130 ka is what Bacon was GIVEN for this depth (row 16,
# `d18O-1`, of DSDP480.csv); 124.853 ka is what Bacon RETURNED for it. Everything else in panel C
# -- the SH82 and KJ90 curves, the pollen, the SH82 markers -- is placed on the model, so plotting
# this one point at its input age mixed input space and output space on a single axis. That is the
# same error the 14C markers had. Both are now shown: the filled marker at the model age with the
# model's 2-sigma, and an open marker at the input tie, so the comparison stays visible.
#
# The model is barely constrained at this depth -- 2-sigma spans 110.8-129.9 ka, 19.1 ka wide,
# against 0.75 ka at 1051 cm where there is radiocarbon control. The old `xerr=0` asserted an
# exact age and was more misleading than the choice of which age to use.
# (A legacy MATLAB value of 126.833 ka also exists. It is the posterior of a SUPERSEDED age model
# -- on the canonical model 126.833 ka falls at 4940 cm, not 4839 -- so it is not used here.)
d18o_ts = {
    'depth':     4839,                     # cm
    'd18o':      np.array([2.62, 2.43]),   # replicate analyses of one sample
    'err':       0.05,                     # analytical, per mil
    'input_age': 130.0,                    # ka, as handed to Bacon
    'input_err': 2.0,                      # ka, 1 sigma
}
d18o_ts['age']    = float(_on_agemodel(median480.values, d18o_ts['depth'])) / 1000
d18o_ts['age_lo'] = float(_on_agemodel(p480[0],          d18o_ts['depth'])) / 1000
d18o_ts['age_hi'] = float(_on_agemodel(p480[3],          d18o_ts['depth'])) / 1000

# Two different things share this Dataset, which is worth knowing before reading it:
#   `lr04_d18o` -- LR04 stack values at the TARGET ages of three Bacon tie points
#                  (38, 69, 130 ka = rows SH-1, SH-2, d18O-1 of DSDP480.csv).
#   `sh_d18o`   -- Shackleton & Hall 1982 measured values at three of THEIR OWN sample depths
#                  (23.95, 34.45, 47.90 m = 2395, 3445, 4790 cm).
# The two lists are not row-aligned and are not meant to be.
tie_points['d18o']= xr.Dataset({"lr04_d18o": (("age"), [4.41, 4.47, 3.67]),
                                "sh_d18o": (("depth"), [3.6, 3.97, 2.48])},
                               coords={"age": [38, 69, 130],
                                       "depth": [23.95, 34.45, 47.90]})
# Place the three SH82 measurements on OUR age model -> [39.395, 71.990, 123.611] ka.
#
# The depths below are SH82 SAMPLE depths, verified against ShackletonHall82_d18O.xlsx: rows at
# 2395, 3445 and 4790 cm carry exactly the d18o values 3.6, 3.97 and 2.48 listed above. Using
# them here is correct and deliberate.
#
# Do not "correct" 4790 to 4839 to match row 16 of DSDP480.csv. Those are two different tie
# points at two different depths: 4839 cm (48.39 m) is this study's benthic d18O tie, and it is
# already represented by `d18o_ts` above; 4790 cm is SH82's deepest measurement. The
# first two Bacon tie points (SH-1, SH-2) happen to sit at SH82 sample depths, the third does not.
#
# The gap these markers are meant to show is in AGE, not depth: SH82 assigned their 4790 cm
# sample an age of 126.082 ka on their own 1982 age model, whereas our Bacon model puts it at
# 123.611 ka -- a ~2.5 ka revision. That is the point of plotting them.
d480_ages = new_median.interp(depth=[2395, 3445, 4790]).to_array(name='age').squeeze()/1000
tie_points['dD']= xr.Dataset({"lr04_d18o": (("age"), [4.12,3.1,3.14,3.67,4.86]),
                                "dsdp480_dD": (("depth"), [-157.26, -142.15, -141.77, -136.01, -149.6, -159.45])},
                               coords={"age": [109,123,125,130,135],
                                       "depth": [37.11, 41.81, 43.15, 45.96, 46.15, 48.11]})

Honestly, I'm not sure how I got the pollen all on one age model before. I need a combined age-depth model for DSDP 480 & 479, but it's unclear to me how to combine the depths for the two cores appropriately. Based on an old matlab code, I suggest that the 479 depths are 8.3 m (830 cm) higher than 480 such that adding 8.3 m to the 479 depths will bring them in line with 480. 

Also, I don't know what the deal is with the SH82 tie-points. Because all the timeseries are plotted vs. age, they should overlap precisely if there is a perfect match. But, given the 2000 year uncertainty range that we used for the Bacon model, there is some offset. But, previously I had the SH d18O measurement from 47.9 plotting right nearby the d18O data produced as part of this study. I don't know how I did that.

especially how I had the MIS5 one plotting right with our new data before. 

In [ ]:
#d480_ages = 
median479.interp(depth=[2395, 3445, 4790])/1000 #.to_array(name='age').squeeze()/1000 #[39.395,  71.990  , 123.611]


In [ ]:
# interp d18O and pollen depths to age model
sh_d18o['age']=new_median.interp(depth=sh_d18o['dat'].depth).to_array(name='age').squeeze()
kj_d18o['age']=new_median.interp(depth=kj_d18o['dat'].depth).to_array(name='age').squeeze()
by_aj['age']=new_median.interp(depth=by_aj['dat'].depth).to_array(name='age').squeeze()

In [ ]:
line_kw={'ls':'-', 'lw':1.5} #, 'marker':'s', 'markersize':4, 'mec':'k', 'mew':0.25, 'zorder':100} 
patch_kw = {'ec':'k', 'lw':1, 'linestyle':':', 'fc':'grey', 'alpha':0.25}
scat_kw = {'s': 40, 'edgecolors':'k', 'alpha':1, 'zorder': 100, 'clip_on': False}
# Filled markers carry a black edge. A marker drawn at the age the MODEL returned (rather than
# the age the model was given) is unfilled: a thin x / + in panels A and B, an open circle in
# panel C. For thin x/+ the stroke colour comes from `color`, so xmark_kw sets no edgecolors.
xmark_kw = {'s': 55, 'linewidths':1.8, 'alpha':1, 'clip_on': False, 'zorder':101}
xtie_kw  = {**scat_kw, 's': 75}   # filled X / P tie-points read too small at s=40
pmark_kw = {**xmark_kw, 's': 70}  # a thin '+' reads smaller than a thin 'x' at equal s
tri_kw   = {**scat_kw, 's': 62}   # ditto the 14C triangles
open_kw  = {'s': 46, 'facecolors':'none', 'edgecolors':'red', 'linewidths':1.6, 'alpha':1, 'clip_on': False, 'zorder':101}
tkw = {'axis':'y', 'direction':'out', 'labelsize': 10}
arrow_kw={'arrowstyle':'->', 'color':'grey', 'linewidth':1, 'clip_on':False}
title_text_kw={'size':14, 'weight':'bold', 'color':'k', 'va':'center'}
label_text_kw={'size':7, 'weight':'bold', 'color':'grey', 'ha':'left', 'va':'center'} #'backgroundcolor':'white', 
laxis_text_kw={'weight':'normal', 'rotation':90, 'size':11, 'color':'k'}
raxis_text_kw={'weight':'normal', 'rotation':270, 'size':11, 'color':'k'}
sh_line_kw = {**line_kw, 'ls':'-.'}   # SH82 dash-dot
kj_line_kw = {**line_kw, 'ls':':'}    # KJ90 dotted
legend_kw = {'loc':'lower center', 'bbox_to_anchor':(0.586, 0.01), 'ncol':3, 'fontsize':8,
             'labelcolor':'k', 'frameon':False, 'columnspacing':1.2, 'handletextpad':0.6}
#settings
xmin=0
xmax=145



fig = plt.figure(figsize=(9,5))

# DSDP-480
ax1 = plt.subplot(111)
#ax.text(1, -23, 'DSDP-480/479', **title_text_kw)
pp=plt.Rectangle((0, hol_d18o_min), 11.7, hol_d18o_max-hol_d18o_min, zorder=100, label='_Hidden', clip_on=True, **patch_kw) 
ax1.add_patch(pp)
# Grey leader line: the arrow touches the Holocene box at 11.7 ka, text sits to its right.
ax1.annotate('Expected Holocene\n$\mathbf{\delta^{18}}$O$\mathbf{_{benthic}}$ for\nGuaymas Basin',
             xy=(11.7,2.2), xytext=(25,2.4),
             arrowprops=arrow_kw, **label_text_kw)
#ax1.annotate('Expected\nHolocene\n$\mathbf{\delta^{18}}$O$\mathbf{_{benthic}}$',
#             xy=(0,hol_d18o_min), xytext=(0,hol_d18o_max-.05),
#             arrowprops=arrow_kw, **label_text_kw, zorder=100)
ax1.plot(sh_d18o['age']/1000, sh_d18o['dat'], c='lightblue', **sh_line_kw, label='SH82 $\delta^{18}$O$_{benthic}$')
ax1.plot(kj_d18o['age']/1000, kj_d18o['dat'], c='lightsteelblue', **kj_line_kw, label='KJ90 $\delta^{18}$O$_{benthic}$')
ax1.plot(lr04['age'], lr04['dat'], c='k', lw=2, label='LR04 $\delta^{18}$O$_{benthic}$') # benthic stack
# Open red circle = a d18O sample drawn at the age the MODEL returned for its depth, plotted
# without error bars. The model's 2-sigma at this depth is 19.1 ka wide and the 130 ka age Bacon
# was GIVEN for it are both still recorded in d18o_ts and in the age-model cell; neither is drawn
# in this panel. Panel A still shows the tie-point itself as a filled circle.
ax1.scatter([d18o_ts['age']]*2, d18o_ts['d18o'],
            marker='o', **open_kw, label='$\\delta^{18}$O$_{benthic}$ tie-point (modeled age)')
# SH82's own d18O samples placed on our age model. Same category as the open circle above
# (a d18O sample at its model age), so same symbol and no separate legend entry.
ax1.scatter(d480_ages, tie_points['d18o'].sh_d18o,
            marker='o', **open_kw, label='_Hidden')
# LR04 value at each tie age -- a reference value on the black LR04 curve, not a tie in our core
ax1.scatter(tie_points['d18o'].lr04_d18o.age, tie_points['d18o'].lr04_d18o,
            marker='o', color='red', s=18, edgecolors='none', zorder=100,
            clip_on=False, label='LR04 value at tie-point (input age)')
ax1.set(xlim=[xmin,xmax], ylim=[5.2,2.1])
ax1.tick_params(color='k', labelcolor='k', top=False, bottom=False, **tkw)
ax1.tick_params(color='k', labelcolor='k', top=False, bottom=False, **tkw)
ax1.minorticks_on()
ax1.set_ylabel(u'$\delta^{18}O_{benthic}$ [‰]', labelpad=5, **laxis_text_kw)
ax1.set_xlabel('AGE (ka)')

ax2=ax1.twinx() 
ax2.plot(by_aj['age']/1000, by_aj['dat'], c='grey', ls='-', lw=1.25, label='Byrne90') # art+jun pollen
ax2.set(xlim=[xmin,xmax], ylim=[33.5,-2], yticks=[0,10,20,30])
ax2.minorticks_on()
ax2.set_ylabel(u'%Artemisia+Juniper', labelpad=15, **raxis_text_kw)
ax2.patch.set_visible(False)
#ax2.spines['bottom'].set_color('none')

# ax2 is a twinx, so its artists are NOT in ax1's handle list -- without merging, the grey
# Byrne90 pollen line is missing from the legend even though it is drawn.
# Explicit legend order: SH82, KJ90, Byrne90, LR04, then the two marker types. matplotlib
# fills a multi-column legend column-major, so with ncol=3 this reads
#   SH82 / KJ90  |  Byrne90 / LR04  |  d18O tie-point / LR04 value
# ax2 is a twinx, so its Byrne90 handle has to be merged in by hand.
h1, lb1 = ax1.get_legend_handles_labels()
h2, lb2 = ax2.get_legend_handles_labels()
_h  = [h1[0],  h1[1],  h2[0],  h1[2],  h1[3],  h1[4]]
_lb = [lb1[0], lb1[1], lb2[0], lb1[2], lb1[3], lb1[4]]
ax1.legend(_h, _lb, **legend_kw)

## gridspec test

In [ ]:
line_kw={'ls':'-', 'lw':2}
err_line_kw={'ls':':', 'lw':0.75, 'color':'k'}
scat_kw = {'s': 40, 'edgecolors':'k', 'alpha':1, 'clip_on': False, 'zorder':100}
# Filled markers carry a black edge. A marker drawn at the age the MODEL returned (rather than
# the age the model was given) is unfilled: a thin x / + in panels A and B, an open circle in
# panel C. For thin x/+ the stroke colour comes from `color`, so xmark_kw sets no edgecolors.
xmark_kw = {'s': 55, 'linewidths':1.8, 'alpha':1, 'clip_on': False, 'zorder':101}
xtie_kw  = {**scat_kw, 's': 75}   # filled X / P tie-points read too small at s=40
pmark_kw = {**xmark_kw, 's': 70}  # a thin '+' reads smaller than a thin 'x' at equal s
tri_kw   = {**scat_kw, 's': 62}   # ditto the 14C triangles
open_kw  = {'s': 46, 'facecolors':'none', 'edgecolors':'red', 'linewidths':1.6, 'alpha':1, 'clip_on': False, 'zorder':101}
tkw = {'axis':'both', 'direction':'in', 'labelsize': 10}
title_text_kw={'size':15, 'weight':'bold', 'color':'k', 'va':'top', 'ha':'right'}
label_text_kw={'size':11, 'weight':'bold', 'color':'firebrick', 'ha':'center', 'va':'bottom'} #'backgroundcolor':'white', 
axis_text_kw={'weight':'normal', 'size':11, 'color':'k'}
legend_kw = {'loc':3, 'fontsize':7, 'labelcolor':'k', 'frameon':False}

fig = plt.figure(figsize=(8,9), constrained_layout=True)

# DSDP-480
ax = plt.subplot2grid((3, 2), (0, 0), rowspan=2, colspan=1)
ax.text(-5000, -300, 'a', **title_text_kw)
# 95% confidence interval
ax.plot(p480[0], dsdp480_agedepth.depth, **err_line_kw, label='_Hidden')
ax.plot(p480[3], dsdp480_agedepth.depth, **err_line_kw, label='_Hidden')
# 2-sigma shading
ax.fill_betweenx(dsdp480_agedepth.depth,
                 p480[0],
                 p480[3],
                 color='k', edgecolor='none', alpha=0.1, label='2$\sigma$ confidence')
# 1-sigma shading
ax.fill_betweenx(dsdp480_agedepth.depth,
                 p480[1],
                 p480[2],
                 color='k', edgecolor='none', alpha=0.2, label='1$\sigma$ confidence')               
# median line
ax.plot(median480, dsdp480_agedepth.depth, '-', c='k', lw=1, label='median')
# ms tie-points
plt.errorbar(d480_tie_age[1:6], dsdp480_input.iloc[1:6].depth.values, 
             xerr=d480_tie_xerr[:, 1:6], yerr=0, fmt='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(d480_tie_age[1:6], dsdp480_input.iloc[1:6].depth.values,
            marker='s', color='lightblue', **scat_kw, label='M.S. tie-point')
# planktic 14C, drawn at Bacon's calibrated (calendar) age -- NOT the raw 14C age in the csv.
# See the age-model cell for why, and for what this does and does not demonstrate.
plt.errorbar(d480_tie_age[6:12], dsdp480_input.iloc[6:12].depth.values, 
             xerr=d480_tie_xerr[:, 6:12], yerr=0, fmt='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(d480_tie_age[6:12], dsdp480_input.iloc[6:12].depth.values,
            marker='^', color='white', **tri_kw, label='$^{14}$C$_{planktic}$, calibrated')
# benthic d18O ties. Rows 12-13 are SH-1/SH-2 (correlated to Shackleton & Hall 1982); row 16 is
# this study's own sample at 4839 cm, which had no legend entry at all before. Same evidence type
# and same role in the model, so one symbol and one legend entry. Row 16 is the same sample that
# appears in panel C.
_d18o_ties = [12, 13, 16]
plt.errorbar(d480_tie_age[_d18o_ties], dsdp480_input.iloc[_d18o_ties].depth.values, 
             xerr=d480_tie_xerr[:, _d18o_ties], yerr=0, fmt='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(d480_tie_age[_d18o_ties], dsdp480_input.iloc[_d18o_ties].depth.values,
            marker='o', color='lightblue', **scat_kw, label='$\delta^{18}$O$_{benthic}$ tie-point')
# Cross-core ties at the age they were GIVEN to Bacon. Row 14 (4307 cm) is the pollen
# correlation and row 15 (4596 cm) the dDwax one -- previously both were drawn as one group
# labelled 'dD_C30 tie', which hid the fact that 4307 is a pollen tie.
plt.errorbar(d480_tie_age[14:16], dsdp480_input.iloc[14:16].depth.values, 
             xerr=d480_tie_xerr[:, 14:16], yerr=0, fmt='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(d480_tie_age[15], dsdp480_input.iloc[15].depth,
            marker='X', color='lightblue', **xtie_kw, label='$\delta$D$_{C30}$ tie-point')
plt.scatter(d480_tie_age[14], dsdp480_input.iloc[14].depth,
            marker='P', color='lightblue', **xtie_kw, label='pollen tie-point')
# The same two ties at their MODEL age
plt.scatter(d480_479_tiepoint[1][0], d480_479_tiepoint[1][1],
            marker='x', color='r', **xmark_kw, label='$\delta$D$_{C30}$ correlation 480↔479')
plt.scatter(d480_479_tiepoint[0][0], d480_479_tiepoint[0][1],
            marker='+', color='coral', **pmark_kw, label='pollen correlation 480↔479')
# Flip y-axis if needed (e.g., for depth increasing downward)
ax.invert_yaxis()
# Labels
ax.text(135000, 100, 'DSDP 480', **title_text_kw)
ax.set(xlim=[1,137000], ylim=[5000,0])
ax.set_xlabel("Age (ka)", **axis_text_kw)
ax.set_ylabel("Core Depth (cm)")
ax.set_xticks([25000,50000,75000,100000,125000])
ax.set_xticklabels([25,50,75,100,125])
ax.xaxis.set_ticks_position('top')    # show ticks on top
ax.xaxis.set_label_position('top')    # show label on top
# Hide bottom ticks and label
ax.tick_params(bottom=False, labelbottom=False, **tkw)
ax.legend(**legend_kw)


# DSDP-479
ax = plt.subplot2grid((3, 2), (0, 1), rowspan=2, colspan=1) #plt.subplot(122)
ax.text(97500, 3290, 'b', **title_text_kw)
# 2-sigma bounds and shading
ax.plot(p479[0], dsdp479_agedepth.depth, **err_line_kw, label='_Hidden')
ax.plot(p479[3], dsdp479_agedepth.depth, **err_line_kw, label='_Hidden')
ax.fill_betweenx(dsdp479_agedepth.depth,
                 p479[0],
                 p479[3],
                 color='k', edgecolor='none', alpha=0.1, label='2$\sigma$ confidence interval')
# 1-sigma shading
ax.fill_betweenx(dsdp479_agedepth.depth,
                 p479[1],
                 p479[2],
                 color='k', edgecolor='none', alpha=0.2, label='1$\sigma$ confidence interval')
# median line
ax.plot(median479, dsdp479_agedepth.depth, '-', c='k', lw=1, label='median')
# d479-LR04 tie points
# DSDP-479's own tie points, at the ages given to Bacon. (dR is 0 for every row here, so the
# old `age - dR` was a no-op.)
plt.errorbar(dsdp479_input.age.values, dsdp479_input.depth.values, 
             xerr=xerr479*2, yerr=0, fmt='none', ecolor='k', capsize=2,label='_Hidden')
plt.scatter(dsdp479_input.age.values, dsdp479_input.depth.values,
            marker='X', color='lightblue', **xtie_kw, label='$\delta$D$_{C30}$ tie-point')
# The DSDP-480 correlations at their MODEL age
plt.scatter(d479_480_tiepoint[1][0], d479_480_tiepoint[1][1],
            marker='x', color='r', **xmark_kw, label='$\delta$D$_{C30}$ correlation 480↔479')
plt.scatter(d479_480_tiepoint[0][0], d479_480_tiepoint[0][1],
            marker='+', color='coral', **pmark_kw, label='pollen correlation 480↔479')
# Flip y-axis if needed (e.g., for depth increasing downward)
ax.invert_yaxis()
# Labels
ax.text(159000, 3490, 'DSDP 479', **title_text_kw)
ax.set(xlim=[100000,160000], ylim=[5750,3450])
ax.set_xlabel("Age (ka)", **axis_text_kw)
ax.yaxis.set_label_position('right')
ax.set_ylabel("Core Depth (cm)", rotation=270, labelpad=20)
ax.set_xticks([100000,120000,140000,160000])
ax.set_xticklabels([100,120,140,160])
ax.xaxis.set_ticks_position('top')    # show ticks on top
ax.xaxis.set_label_position('top')    # show label on top
# Hide bottom ticks and label
ax.tick_params(bottom=False, labelbottom=False,
               left=False, labelleft=False,
               right=True, labelright=True, **tkw)



### Timeseries
line_kw={'ls':'-', 'lw':1.5} #, 'marker':'s', 'markersize':4, 'mec':'k', 'mew':0.25, 'zorder':100} 
patch_kw = {'ec':'k', 'lw':1, 'linestyle':':', 'fc':'grey', 'alpha':0.25}
scat_kw = {'s': 40, 'edgecolors':'k', 'alpha':1, 'zorder': 100, 'clip_on': False}
# Filled markers carry a black edge. A marker drawn at the age the MODEL returned (rather than
# the age the model was given) is unfilled: a thin x / + in panels A and B, an open circle in
# panel C. For thin x/+ the stroke colour comes from `color`, so xmark_kw sets no edgecolors.
xmark_kw = {'s': 55, 'linewidths':1.8, 'alpha':1, 'clip_on': False, 'zorder':101}
xtie_kw  = {**scat_kw, 's': 75}   # filled X / P tie-points read too small at s=40
pmark_kw = {**xmark_kw, 's': 70}  # a thin '+' reads smaller than a thin 'x' at equal s
tri_kw   = {**scat_kw, 's': 62}   # ditto the 14C triangles
open_kw  = {'s': 46, 'facecolors':'none', 'edgecolors':'red', 'linewidths':1.6, 'alpha':1, 'clip_on': False, 'zorder':101}
tkw = {'axis':'y', 'direction':'out', 'labelsize': 10}
arrow_kw={'arrowstyle':'->', 'color':'grey', 'linewidth':1, 'clip_on':False}
title_text_kw={'size':14, 'weight':'bold', 'color':'k', 'va':'center'}
label_text_kw={'size':7, 'weight':'bold', 'color':'grey', 'ha':'left', 'va':'center'} #'backgroundcolor':'white', 
laxis_text_kw={'weight':'normal', 'rotation':90, 'size':11, 'color':'k'}
raxis_text_kw={'weight':'normal', 'rotation':270, 'size':11, 'color':'grey'}
sh_line_kw = {**line_kw, 'ls':'-.'}   # SH82 dash-dot
kj_line_kw = {**line_kw, 'ls':':'}    # KJ90 dotted
legend_kw = {'loc':'lower center', 'bbox_to_anchor':(0.586, 0.01), 'ncol':3, 'fontsize':7,
             'labelcolor':'k', 'frameon':False, 'columnspacing':1.2, 'handletextpad':0.6}


ax1 = plt.subplot2grid((3, 2), (2, 0), rowspan=1, colspan=2)
ax1.text(-7.5, 2.1, 'c', **title_text_kw)
pp=plt.Rectangle((0, hol_d18o_min), 11.7, hol_d18o_max-hol_d18o_min, zorder=100, label='_Hidden', clip_on=True, **patch_kw) 
ax1.add_patch(pp)
# Grey leader line: the arrow touches the Holocene box at 11.7 ka, text sits to its right.
ax1.annotate('Expected Holocene\n$\mathbf{\delta^{18}}$O$\mathbf{_{benthic}}$ for\nGuaymas Basin',
             xy=(11.7,2.2), xytext=(25,2.4),
             arrowprops=arrow_kw, **label_text_kw)
#ax1.annotate('Expected\nHolocene\n$\mathbf{\delta^{18}}$O$\mathbf{_{benthic}}$',
#             xy=(0,hol_d18o_min), xytext=(0,hol_d18o_max-.05),
#             arrowprops=arrow_kw, **label_text_kw, zorder=100)
ax1.plot(sh_d18o['age']/1000, sh_d18o['dat'], c='lightblue', **sh_line_kw, label='SH82')
ax1.plot(kj_d18o['age']/1000, kj_d18o['dat'], c='lightsteelblue', **kj_line_kw, label='KJ90')
ax1.plot(lr04['age'], lr04['dat'], c='k', lw=2, label='LR04') # benthic stack
# Open red circle = a d18O sample drawn at the age the MODEL returned for its depth, plotted
# without error bars. The model's 2-sigma at this depth is 19.1 ka wide and the 130 ka age Bacon
# was GIVEN for it are both still recorded in d18o_ts and in the age-model cell; neither is drawn
# in this panel. Panel A still shows the tie-point itself as a filled circle.
ax1.scatter([d18o_ts['age']]*2, d18o_ts['d18o'],
            marker='o', **open_kw, label='$\\delta^{18}$O$_{benthic}$ tie-point (modeled age)')
# SH82's own d18O samples placed on our age model. Same category as the open circle above
# (a d18O sample at its model age), so same symbol and no separate legend entry.
ax1.scatter(d480_ages, tie_points['d18o'].sh_d18o,
            marker='o', **open_kw, label='_Hidden')
# LR04 value at each tie age -- a reference value on the black LR04 curve, not a tie in our core
ax1.scatter(tie_points['d18o'].lr04_d18o.age, tie_points['d18o'].lr04_d18o,
            marker='o', color='red', s=18, edgecolors='none', zorder=100,
            clip_on=False, label='LR04 value at tie-point (input age)')
ax1.set(xlim=[xmin,xmax], ylim=[5.2,2.1])
ax1.tick_params(color='k', labelcolor='k', top=False, bottom=False, **tkw)
ax1.tick_params(color='k', labelcolor='k', top=False, bottom=False, **tkw)
ax1.minorticks_on()
ax1.set_ylabel(u'$\delta^{18}O_{benthic}$ [‰]', labelpad=5, **laxis_text_kw)
ax1.set_xlabel('AGE (ka)')

ax2=ax1.twinx() 
ax2.plot(by_aj['age']/1000, by_aj['dat'], c='grey', ls='-', lw=1.25, label='Byrne90') # art+jun pollen
ax2.set(xlim=[xmin,xmax], ylim=[33.5,-2], yticks=[0,10,20,30])
ax2.minorticks_on()
ax2.tick_params(color='grey', labelcolor='grey', top=False, **tkw)
ax2.set_ylabel(u'%Artemisia+Juniper', labelpad=15, **raxis_text_kw)
ax2.patch.set_visible(False)
ax2.spines['right'].set_color('grey')

# ax2 is a twinx, so its artists are NOT in ax1's handle list -- without merging, the grey
# Byrne90 pollen line is missing from the legend even though it is drawn.
# Explicit legend order: SH82, KJ90, Byrne90, LR04, then the two marker types. matplotlib
# fills a multi-column legend column-major, so with ncol=3 this reads
#   SH82 / KJ90  |  Byrne90 / LR04  |  d18O tie-point / LR04 value
# ax2 is a twinx, so its Byrne90 handle has to be merged in by hand.
h1, lb1 = ax1.get_legend_handles_labels()
h2, lb2 = ax2.get_legend_handles_labels()
_h  = [h1[0],  h1[1],  h2[0],  h1[2],  h1[3],  h1[4]]
_lb = [lb1[0], lb1[1], lb2[0], lb1[2], lb1[3], lb1[4]]
ax1.legend(_h, _lb, **legend_kw)

#plt.savefig(f'{opath}/dsdp_480-479.agemodel.pdf', bbox_inches='tight')

### Figure caption

Revised from the manuscript to match the marker and line scheme actually plotted above.
Changes from the submitted version are listed after the block.

```latex
\textbf{Age-depth model for DSDP Sites 480 and 479.} BACON-generated age models for
(a) Site 480, based on 6 accelerated mass spectrometer $^{14}$C measurements of mixed
\textit{Globigerina bulloides} and \textit{Neogloboquadrina dutertrei} planktic foraminifera
\cite<white triangles, plotted at their BACON-calibrated calendar ages, >{keigwin90p},
magnetic secular correlations \cite<blue squares, >{barron04mm}, benthic $\delta^{18}$O
stratigraphy \cite<blue circles, >[ this study]{shackleton82irddp}, and correlation of
palynological \cite<blue plus sign, >{byrne90} and \dDw\ (blue x, this study) records to
Site 479; those same two correlations are also shown at the ages returned by the age model
(orange plus sign and red x, respectively). (b) Site 479, based on tie-points between local
\dDw\ (blue x's) and the LR04 $\delta^{18}O_{benthic}$ stack \cite{lisiecki05p}, with the
Site 480 correlations at their modelled ages (orange plus sign, red x). Shading in (a) and
(b) gives the 1$\sigma$ and 2$\sigma$ credible intervals of the BACON ensemble about its
median (solid line); horizontal bars are the 2$\sigma$ uncertainty on each tie-point.
(c) Time-series of DSDP Site 480 $\delta^{18}O_{benthic}$ measurements from
\citeA{shackleton82irddp} (dash-dotted light blue) and \citeA{keigwin90p} (dotted light
blue), new $\delta^{18}O_{benthic}$ measurements from this study (open red circles, plotted
at the ages returned by the age model), and the Site 480 Artemisia and Juniper pollen record
of \citeA{byrne90} (solid grey), all placed on the combined Site 480/479 age model generated
here and compared to the LR04 $\delta^{18}O_{benthic}$ stack \cite<black, >{lisiecki05p}.
Red dots mark the LR04 value at the age assigned to each tie-point. The grey box indicates
the expected Holocene $\delta^{18}O_{benthic}$ range for Guaymas Basin.
```

**What changed, and why**

| Submitted | Revised | Reason |
|---|---|---|
| benthic δ¹⁸O stratigraphy = *blue triangles* | *blue circles* | triangles now mean radiocarbon only; a circle always means benthic δ¹⁸O |
| palynological correlation = *orange x's* | *orange plus sign* | x is reserved for the δD<sub>wax</sub> correlation, so the two are no longer both x's |
| ¹⁴C given without a timescale | "plotted at their BACON-calibrated calendar ages" | the values in `DSDP480.csv` are uncalibrated ¹⁴C years and used to be drawn directly against a calendar-year model |
| — | blue x / blue plus sign added to (a) | tie-points *as given to* BACON are now drawn separately from the same correlations at their *modelled* ages |
| — | line styles named in (c) | SH82 and KJ90 are now distinguished by dash pattern, not colour alone |
| "Site 480/479 ... pollen record" in (c) | **"Site 480"** | see the caveat below |

**Caveat on the pollen record in (c).** The Byrne90 record spans 10–12150 cm, but the Site 480
age model only covers 18–4944 cm, so 41 of its 130 samples interpolate to NaN and are never
drawn. The deepest sample actually plotted sits at 4890 cm ≈ 125.9 ka. Calling the plotted curve
the "Site 480/479" record would overstate it, so the caption says Site 480. Extending it past
125.9 ka requires deciding how to age the deeper pollen samples — it is not a plotting fix.